In [1]:
"""
User:
----------
Age (numeric)
G (F,M)
TK (String)

Rest:
----------
Cuisine: (Pizza, Burger, Pasta, Souvlaki, Sushi, Chinese)
PriceRange: ($,$$,$$$,$$$$)
HAS_SELF_DEL: B
MIN_COST: (numeric)
AVG_RATING: (numeric)
AVG_DEL_TIME: (numeric)
HAS_OFFER: (b)
HAS_EXTRA_DEL_COST (b)
PAYMENT_METHODS: (CASH, CREDIT, COUPONS)


TARGET:0/1

SEGMENT 1: One trick pony  
--------------------------------
* Single cuisine lover 
* Age: Gaussian distro with a mean of 60
* AVG_RATING >4
* AVG_DEL_TIME <= 35 min
* HAS_OFFER: 60% influence

"""

'\nUser:\n----------\nAge (numeric)\nG (F,M)\nTK (String)\n\nRest:\n----------\nCuisine: (Pizza, Burger, Pasta, Souvlaki, Sushi, Chinese)\nPriceRange: ($,$$,$$$,$$$$)\nHAS_SELF_DEL: B\nMIN_COST: (numeric)\nAVG_RATING: (numeric)\nAVG_DEL_TIME: (numeric)\nHAS_OFFER: (b)\nHAS_EXTRA_DEL_COST (b)\nPAYMENT_METHODS: (CASH, CREDIT, COUPONS)\n\n\nTARGET:0/1\n\nSEGMENT 1: One trick pony  \n--------------------------------\n* Single cuisine lover \n* Age: Gaussian distro with a mean of 60\n* AVG_RATING >4\n* AVG_DEL_TIME <= 35 min\n* HAS_OFFER: 60% influence\n\n'

In [2]:
import random
import csv
import numpy as np
from dataclasses import dataclass



In [3]:
[1, 3].index(1)

0

In [4]:
@dataclass
class Restaurant:
    cuisine: list
    price_range: str
    has_self_del: bool
    has_offer: bool
    has_extra_del_cost: bool
    min_cost: float
    avg_rating: float
    avg_del_time: int
    payment_methods: list


In [5]:
np.random.multinomial(1, [0.5, 0.3, 0.15, 0.05])

array([0, 0, 1, 0])

In [6]:
def generate_restaurants(rest_num: int = 100):
    restaurants = []

    for i in range(rest_num):
        cuisine_num = random.randint(1, 3)

        cuisine = random.sample(['Pizza', 'Burger', 'Pasta',
                                 'Souvlaki', 'Sushi', 'Chinese'], cuisine_num)

        price_sample = np.random.multinomial(1, [0.5, 0.3, 0.15, 0.05]).argmax()
        price_range = random.choice(["$", "$$", "$$$", "$$$$"][price_sample])

        has_self_del = random.choice([True, False])

        has_offer = random.choice([True, False])

        has_extra_del_cost = True if random.random() < 0.2 else False

        min_cost = round(random.gauss(6, 3), 1)

        avg_rating = round(random.gauss(3.7, 1), 1)

        avg_del_time = round(random.gauss(30, 7))

        payment_methods = random.sample(["CASH", "CARD"], 2) + random.randint(0, 1) * ['COUPON']

        restaurants.append(
            Restaurant(cuisine, price_range, has_self_del,
                       has_offer, has_extra_del_cost, min_cost,
                       avg_rating, avg_del_time, payment_methods)
        )

    return restaurants

In [7]:
restaurants = generate_restaurants(rest_num=100)

In [8]:
restaurants[7].cuisine

['Burger']

In [9]:
@dataclass
class User:
    gender: str
    age: int
    pcode: str
    favorite_cuisine: list


In [10]:

def generate_users_segment1(user_num: int = 1000,
                            postcode_num: int = 50
                            ):
    """
    SEGMENT 1: One trick pony  
    --------------------------------
    * Single cuisine lover 
    * Age: Gaussian distro with a mean of 60
    * AVG_RATING >4
    * AVG_DEL_TIME <= 35 min
    * HAS_OFFER: 60% influence
    * 90% consistent
    """
    postcode_list = ['PC' + str(i) for i in range(postcode_num)]

    users = []

    for i in range(user_num):
        gender = random.choice(["M", "F"])  # pick a random gender

        pcode = random.choice(postcode_list)  # pick a random postcode

        age = int(np.random.normal(loc=60, scale=7, size=1)[0])  # random.randint(18,100) # pick a random age

        #pick random single favorite cuisine
        fave = random.sample(['Pizza', 'Burger', 'Pasta',
                              'Souvlaki', 'Sushi', 'Chinese'], 1)

        #append the new user
        users.append(User(gender, age, pcode, fave))

    return users

    #=======================================



In [11]:

def generate_ratings_segment1(
        users: list,
        restaurants: list
):
    """
    SEGMENT 1: One trick pony  
    --------------------------------
    * Single cuisine lover 
    * Age: Gaussian distro with a mean of 60
    * AVG_RATING >4
    * AVG_DEL_TIME <= 35 min
    * HAS_OFFER: 60% influence
    * 90% consistent
    """

    fw = open('segment1.csv', 'w')
    writer = csv.writer(fw)

    writer.writerow(['age', 'gender', 'pcode', 'favorite_cuisine', 'restaurant_cuisine', 'price_range', 'has_self_del',
                     'has_offer', 'has_extra_del_cost', 'min_cost',
                     'avg_rating', 'avg_del_time', 'payment_methods', 'rating'])

    cnt = 0
    total = 0
    for usr in users:  # for each user

        rating_num = random.randint(1, len(restaurants))  # get the number of ratings
        total += rating_num
        my_restaurants = random.sample(restaurants, rating_num)  # sample restaurants to be rated

        for rst in my_restaurants:  # for each restaurant

            rating = 0  # initialize to negative rating

            # if the restaurant has my single favorite cuisine, and a rating over 4 and a delivery time<35 min
            if (usr.favorite_cuisine[0] in rst.cuisine) and (rst.avg_rating > 4) and (rst.avg_del_time < 35):

                # one random 50-50 chance, and one 60% chance if the rest has an offer
                if random.random() or (rst.has_offer and random.random() <= 0.6):
                    rating = 1
                    cnt += 1

            if random.random() < 0.1:  #consistency switch
                rating *= -1

            new_row = [usr.age, usr.gender, usr.pcode, usr.favorite_cuisine, rst.cuisine, rst.price_range,
                       rst.has_self_del, rst.has_offer, rst.has_extra_del_cost,
                       rst.min_cost, rst.avg_rating, rst.avg_del_time,
                       rst.payment_methods, rating]

            writer.writerow(new_row)

    fw.close()

    print('ones', cnt / total)


In [12]:

def generate_users_segment2(user_num: int = 1000,
                            postcode_num: int = 50
                            ):
    """
    SEGMENT 2: young+price-driven  
    -------------------------------- 
    * Diverse set of favorite cuisines, but each person has their fave
    * Age: Gaussian distro with a mean of 20
    * AVG_RATING >4.2 if price in [$$$] 
    * AVG_RATING >3.5 if price in [$$]
    * AVG_RATING >3 if price in [$] 
    * no way if price in [$$$$]
    * no way if has_extra_del_cost
    * HAS_OFFER: 80% influence if price in [$$$], 60% influence if [$,$$]
    * 80% consistent
    """
    postcode_list = ['PC' + str(i) for i in range(postcode_num)]

    users = []

    for i in range(user_num):
        gender = random.choice(["M", "F"])  # pick a random gender

        pcode = random.choice(postcode_list)  # pick a random postcode

        age = int(np.random.normal(loc=20, scale=7, size=1)[0])  # random.randint(18,100) # pick a random age

        #pick random favorite cuisines
        fave = random.sample(['Pizza', 'Burger', 'Pasta',
                              'Souvlaki', 'Sushi', 'Chinese'], random.randint(1, 6))

        #append the new user
        users.append(User(gender, age, pcode, fave))

    return users

    #=======================================



In [13]:

def generate_ratings_segment2(
        users: list,
        restaurants: list,
):
    """
    SEGMENT 2: young+price-driven  
    -------------------------------- 
    * Diverse set of favorite cuisines, but each person has their fave
    * Age: Gaussian distro with a mean of 20
    * AVG_RATING >4.2 if price in [$$$] 
    * AVG_RATING >3.5 if price in [$$]
    * AVG_RATING >3 if price in [$] 
    * no way if price in [$$$$]
    * no way if has_extra_del_cost
    * HAS_OFFER: 80% influence if price in [$$$], 60% influence if [$,$$]
    * 80% consistent
    """

    fw = open('segment2.csv', 'w')
    writer = csv.writer(fw)

    writer.writerow(['age', 'gender', 'pcode', 'favorite_cuisine', 'restaurant_cuisine', 'price_range', 'has_self_del',
                     'has_offer', 'has_extra_del_cost', 'min_cost',
                     'avg_rating', 'avg_del_time', 'payment_methods', 'rating'])

    cnt = 0
    total = 0

    for usr in users:  # for each user

        rating_num = random.randint(1, len(restaurants))  # get the number of ratings
        total += rating_num
        my_restaurants = random.sample(restaurants, rating_num)  # sample restaurants to be rated

        for rst in my_restaurants:  # for each restaurant

            rating = 0  # initialize to negative rating

            # if the restaurant has my single favorite cuisine, and a rating over 4 and a delivery time<35 min
            if (any(element in rst.cuisine for element in usr.favorite_cuisine)) and \
                    (not rst.has_extra_del_cost):

                if (rst.price_range == '$' and rst.avg_rating > 3.5) or \
                        (rst.price_range == '$$' and rst.avg_rating > 4) or \
                        (rst.price_range == '$$$' and rst.avg_rating > 4.5):

                    # one random 50-50 chance, or offer boosts
                    if random.random() or \
                            (rst.has_offer and rst.price_range in ['$', '$$'] and random.random() <= 0.6) or \
                            (rst.has_offer and rst.price_range == '$$$' and random.random() <= 0.8):
                        rating = 1
                        cnt += 1

            if random.random() < 0.2:  #consistency switch
                rating *= -1

            new_row = [usr.age, usr.gender, usr.pcode, usr.favorite_cuisine, rst.cuisine, rst.price_range,
                       rst.has_self_del, rst.has_offer, rst.has_extra_del_cost,
                       rst.min_cost, rst.avg_rating, rst.avg_del_time,
                       rst.payment_methods, rating]

            writer.writerow(new_row)

    fw.close()

    print('ones', cnt / total, cnt, total)


In [17]:


users_segment1 = generate_users_segment1(user_num=1000)
generate_ratings_segment1(users_segment1, restaurants)

users_segment2 = generate_users_segment2(user_num=2000)
generate_ratings_segment2(users_segment2, restaurants)


ones 0.0848448008784486
ones 0.3299737291606317 33034 100111


In [18]:
import pandas as pd

file_paths = ['segment1.csv', 'segment2.csv']  # Add as many paths as needed

# Read each CSV file into a DataFrame and store them in a list
dfs = [pd.read_csv(file_path) for file_path in file_paths]

# Concatenate all DataFrames into a single DataFrame
combined_df = pd.concat(dfs, ignore_index=True)

# Shuffle the rows of the combined DataFrame
df = combined_df.sample(frac=1).reset_index(drop=True)

# shuffled_df now contains all your data, shuffled
print(df.shape)
df

(151110, 14)


,age,gender,pcode,favorite_cuisine,restaurant_cuisine,price_range,has_self_del,has_offer,has_extra_del_cost,min_cost,avg_rating,avg_del_time,payment_methods,rating
0,16,M,PC23,"['Souvlaki', 'Sushi', 'Chinese', 'Burger', 'Pa...",['Chinese'],$,True,False,False,11.0,2.1,24,"['CARD', 'CASH', 'COUPON']",0
1,19,F,PC47,"['Souvlaki', 'Burger']",['Chinese'],$,True,False,False,3.3,4.4,29,"['CASH', 'CARD', 'COUPON']",0
2,70,F,PC32,['Chinese'],['Souvlaki'],$,False,False,False,9.0,4.5,39,"['CASH', 'CARD', 'COUPON']",0
3,26,M,PC11,"['Chinese', 'Souvlaki']","['Pizza', 'Burger']",$,True,False,False,1.9,5.4,40,"['CARD', 'CASH', 'COUPON']",0
4,22,M,PC12,"['Pizza', 'Sushi']","['Pizza', 'Souvlaki', 'Chinese']",$,False,True,False,5.7,3.3,30,"['CARD', 'CASH']",0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
151105,70,F,PC3,['Chinese'],['Chinese'],$,True,False,False,3.3,4.4,29,"['CASH', 'CARD', 'COUPON']",1
151106,61,M,PC11,['Pasta'],['Burger'],$,False,False,False,9.8,4.9,31,"['CARD', 'CASH', 'COUPON']",0
151107,66,M,PC29,['Pasta'],"['Chinese', 'Burger', 'Sushi']",$,True,True,False,3.7,2.7,14,"['CARD', 'CASH']",0
151108,9,F,PC13,"['Souvlaki', 'Burger', 'Pasta']","['Pasta', 'Burger', 'Souvlaki']",$,False,False,False,5.1,4.6,23,"['CARD', 'CASH']",1


In [19]:
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
import ast


# Function to convert string representation of lists to actual lists
def convert_string_to_list(df, column_name):
    df[column_name] = df[column_name].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)


# Convert string representations of lists to actual lists for all list-type columns
list_columns = ['favorite_cuisine', 'restaurant_cuisine', 'payment_methods']
for column in list_columns:
    convert_string_to_list(df, column)

# One-hot encode each list-type column
for column in list_columns:
    mlb = MultiLabelBinarizer()
    expanded = mlb.fit_transform(df[column])
    encoded_df = pd.DataFrame(expanded, columns=[f"{column}_{cls}" for cls in mlb.classes_])
    df = df.join(encoded_df)

# Drop the original list-type columns if they are no longer needed
df.drop(list_columns, axis=1, inplace=True)



In [19]:
!pip install catboost

In [21]:

import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Split the data into features and target
X = df.drop('rating', axis=1)
y = df['rating']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Identify categorical features in the dataset
categorical_features_indices = np.where(X.dtypes != np.float64)[0]

# Initialize CatBoostClassifier
model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.1,
    depth=3,
    eval_metric='Accuracy',
    use_best_model=True,
    random_seed=42,
    verbose=2,
    auto_class_weights='Balanced'  # Helps if the target classes are imbalanced
)

# Train the model
model.fit(
    X_train, y_train,
    cat_features=categorical_features_indices,
    eval_set=(X_test, y_test),
    early_stopping_rounds=50  # Stops training if the validation metric is not improving
)

# Make predictions on the test set
predictions = model.predict(X_test)

# Calculate the accuracy
accuracy = accuracy_score(y_test, predictions)
print(f"Accuracy: {accuracy:.4f}")

0:	learn: 0.5808394	test: 0.5873331	best: 0.5873331 (0)	total: 246ms	remaining: 4m 6s
200:	learn: 0.6811404	test: 0.6613030	best: 0.6618982 (198)	total: 14.7s	remaining: 58.6s
400:	learn: 0.7018770	test: 0.6778375	best: 0.6787471 (371)	total: 19.5s	remaining: 29.1s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.6815180797
bestIteration = 453

Shrink model to first 454 iterations.
Accuracy: 0.8461


In [22]:
X_test

,age,gender,pcode,price_range,has_self_del,has_offer,has_extra_del_cost,min_cost,avg_rating,avg_del_time,...,favorite_cuisine_Sushi,restaurant_cuisine_Burger,restaurant_cuisine_Chinese,restaurant_cuisine_Pasta,restaurant_cuisine_Pizza,restaurant_cuisine_Souvlaki,restaurant_cuisine_Sushi,payment_methods_CARD,payment_methods_CASH,payment_methods_COUPON
73216,26,M,PC36,$,False,True,False,4.2,3.6,24,...,1,0,0,0,0,1,0,1,1,1
65922,72,F,PC35,$,False,True,True,11.8,4.7,19,...,0,0,0,1,0,0,0,1,1,0
48124,61,F,PC0,$,False,False,False,5.2,3.4,20,...,0,0,0,1,0,0,0,1,1,1
60025,17,F,PC36,$,False,False,False,9.8,4.9,31,...,0,1,0,0,0,0,0,1,1,1
142529,56,F,PC43,$,False,False,False,4.6,4.2,44,...,0,0,0,1,0,1,1,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52666,16,F,PC35,$,True,False,False,4.6,2.4,36,...,1,1,0,0,0,0,0,1,1,1
56107,12,F,PC8,$,True,False,False,14.2,5.6,34,...,1,0,0,1,1,0,1,1,1,0
141905,32,F,PC5,$,False,False,True,7.9,5.1,40,...,1,0,0,1,0,0,0,1,1,1
103796,23,F,PC25,$,True,False,False,3.3,4.4,29,...,0,0,1,0,0,0,0,1,1,1
